# PBQuantum Labs — Kaggle T4 QLoRA Fine-Tuning Pipeline

This reproducible notebook fine-tunes an open-weights Instruct LLM (e.g., **Qwen/Qwen2.5-7B-Instruct** or **meta-llama/Meta-Llama-3-8B-Instruct**) on **Kaggle T4 GPU (16GB VRAM)** using **4-bit QLoRA (Quantized Low-Rank Adaptation)**.

### Core Principles
1. **Never Train from Random Initialization**: Starts from a strong instruction-tuned base model.
2. **Dynamic Kaggle Dataset Discovery**: Dynamically scans `/kaggle/input/` without assuming a hardcoded path.
3. **Pedagogical Alignment**: Teaches the model QuantumLab's 9-step learning loop, Dirac bra-kets, clean Qiskit 1.0 code, and strict anti-hallucination abstention.
4. **Baseline vs. Fine-Tuned Evaluation**: Automatically compares and logs `base_results.json` vs `finetuned_results.json`.

In [ ]:
# Step 1: GPU & Environment Verification
import torch
import sys
import os

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("WARNING: No GPU detected! Please switch Kaggle Accelerator to 'GPU T4 x2' or 'GPU T4'.")

In [ ]:
# Step 2: Install Modern Training Stack
!pip install -q --upgrade pip
!pip install -q transformers>=4.44.0 accelerate>=0.33.0 peft>=0.12.0 bitsandbytes>=0.43.0 trl>=0.9.6 datasets>=2.20.0
print("Dependencies successfully installed!")

In [ ]:
# Step 3: Dynamic Dataset Discovery in /kaggle/input/
import glob

kaggle_input = "/kaggle/input"
found_files = []

if os.path.exists(kaggle_input):
    print("Scanning /kaggle/input directories:")
    for root, dirs, files in os.walk(kaggle_input):
        for file in files:
            if file.endswith(".jsonl") or file.endswith(".json") or file.endswith(".txt") or file.endswith(".pdf"):
                full_path = os.path.join(root, file)
                found_files.append(full_path)
                print(f" - Found: {full_path}")
else:
    print("Local / testing environment detected (outside Kaggle).")

print(f"Total potential training artifacts discovered: {len(found_files)}")

In [ ]:
# Step 4: Dataset Assembly & Validation
import json
from datasets import Dataset

training_data = []

# Look for seed_benchmark or user-provided jsonl in discovered files
for fpath in found_files:
    if fpath.endswith(".jsonl"):
        try:
            with open(fpath, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        item = json.loads(line.strip())
                        if "messages" in item:
                            training_data.append(item["messages"])
        except Exception as e:
            print(f"Error reading {fpath}: {e}")

# Fallback curated seed examples if no dataset mounted yet
if not training_data:
    print("No custom dataset mounted yet. Initializing canonical PBQuantum seed dataset...")
    training_data = [
        [
            {"role": "user", "content": "What is a qubit?"},
            {"role": "assistant", "content": "A qubit (quantum bit) is the fundamental unit of quantum information. Unlike a classical bit that is strictly either 0 or 1, a pure qubit state exists as a normalized linear superposition in a two-dimensional complex Hilbert space ℂ²: |ψ⟩ = α|0⟩ + β|1⟩, where α, β ∈ ℂ satisfy the normalization constraint |α|² + |β|² = 1. Geometrically, any pure single-qubit state maps to a point on the 3D unit Bloch sphere parameterized by polar angle θ and azimuthal phase angle φ."}
        ],
        [
            {"role": "user", "content": "Explain superposition. Does it mean a qubit is physically both 0 and 1 at the same time?"},
            {"role": "assistant", "content": "No. Describing a qubit as 'literally both 0 and 1 at once' is a common popular misconception. Physically, a qubit in superposition exists in a definite, well-defined single quantum state vector |ψ⟩ = α|0⟩ + β|1⟩. It is a coherent linear combination of basis vectors possessing definite relative phases. When measured in the computational { |0⟩, |1⟩ } basis, the state collapses probabilistically according to Born's Rule, yielding outcome 0 with probability |α|² and outcome 1 with probability |β|²."}
        ],
        [
            {"role": "user", "content": "Derive the action of the Hadamard gate on the ground state |0⟩."},
            {"role": "assistant", "content": "Let the ground state in vector notation be |0⟩ = [1, 0]ᵀ. The single-qubit Hadamard gate is represented by the 2x2 unitary matrix H = (1/√2) [[1, 1], [1, -1]]. Applying H to |0⟩ via matrix-vector multiplication yields:\n\nH|0⟩ = (1/√2) [[1, 1], [1, -1]] [[1], [0]] = (1/√2) [[1], [1]] = (1/√2)|0⟩ + (1/√2)|1⟩ = |+⟩.\n\nThe measurement probabilities are P(0) = |1/√2|² = 1/2 (50%) and P(1) = |1/√2|² = 1/2 (50%), satisfying total probability normalization 1/2 + 1/2 = 1."}
        ],
        [
            {"role": "user", "content": "Write modern Python Qiskit 1.0 code to create an entangled Bell state |Φ⁺⟩ and compute its statevector."},
            {"role": "assistant", "content": "```python\nfrom qiskit import QuantumCircuit\nfrom qiskit.quantum_info import Statevector\n\nqc = QuantumCircuit(2)\nqc.h(0)\nqc.cx(0, 1)\n\nstate = Statevector.from_instruction(qc)\nprint(\"Bell State |Φ⁺⟩ Statevector:\", state)\nprint(\"Probabilities:\", state.probabilities_dict())\n```\nExpected output: statevector amplitudes for |00⟩ and |11⟩ are both 1/√2 ≈ 0.7071, with probabilities P(00) = 0.5 and P(11) = 0.5."}
        ]
    ]

print(f"Total training conversations ready: {len(training_data)}")
dataset = Dataset.from_dict({"messages": training_data})
split_dataset = dataset.train_test_split(test_size=0.2 if len(training_data) > 5 else 0.0)
print("Dataset prepared successfully.")

In [ ]:
# Step 5: Load Base Model with 4-bit Quantization (QLoRA)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # Alternative: "meta-llama/Meta-Llama-3-8B-Instruct"
print(f"Loading Base Model: {MODEL_ID} with 4-bit quantization on T4 GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model memory footprint: {model.get_memory_footprint() / (1024**3):.2f} GB")

In [ ]:
# Step 6: Baseline Model Evaluation (Before Fine-Tuning)
eval_prompts = [
    "Explain superposition. Does it mean a qubit is physically both 0 and 1 at the same time?",
    "Derive the action of the Hadamard gate on the ground state |0⟩.",
    "Can quantum entanglement transmit messages faster than light?",
    "What was Albert Einstein's private diary entry on December 14, 1928 regarding quantum circuits?"
]

base_results = {}
print("Generating Baseline Responses...")

model.eval()
for prompt in eval_prompts:
    messages = [{"role": "user", "content": prompt}]
    formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.6, do_sample=True)
    
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    base_results[prompt] = reply

os.makedirs("/kaggle/working/eval", exist_ok=True)
with open("/kaggle/working/eval/base_results.json", "w", encoding="utf-8") as f:
    json.dump(base_results, f, indent=2)

print("Baseline evaluation saved to /kaggle/working/eval/base_results.json")

In [ ]:
# Step 7: Configure PEFT / QLoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 8: Configure SFTTrainer
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/quantumlab_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    max_seq_length=1024,
    dataset_text_field=None,
)

def formatting_prompts_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False)

trainer = SFTTrainer(
    model=model,
    train_dataset=split_dataset["train"],
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=formatting_prompts_func,
)

print("Trainer configured. Ready to run trainer.train() on Kaggle GPU T4!")

In [ ]:
# Step 9: Run Supervised Fine-Tuning
# Note: Execute this cell when running interactively in Kaggle notebook!
train_result = trainer.train()
print("Training Complete!")
trainer.save_model("/kaggle/working/quantumlab-qlora-adapter")
tokenizer.save_pretrained("/kaggle/working/quantumlab-qlora-adapter")
print("Adapter saved to /kaggle/working/quantumlab-qlora-adapter")

In [ ]:
# Step 10: Fine-Tuned Model Evaluation & Comparison Logging
finetuned_results = {}
print("Generating Fine-Tuned Model Responses...")

model.eval()
for prompt in eval_prompts:
    messages = [{"role": "user", "content": prompt}]
    formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.6, do_sample=True)
    
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    finetuned_results[prompt] = reply

with open("/kaggle/working/eval/finetuned_results.json", "w", encoding="utf-8") as f:
    json.dump(finetuned_results, f, indent=2)

print("Fine-tuned evaluation saved to /kaggle/working/eval/finetuned_results.json")
print("\n--- Sample Comparison on Misconception Prompt ---")
test_prompt = eval_prompts[0]
print("[Prompt]:", test_prompt)
print("\n[Base Model Output]:\n", base_results.get(test_prompt, "N/A"))
print("\n[Fine-Tuned Output]:\n", finetuned_results.get(test_prompt, "N/A"))